<a href="https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahid-Imran/flyrank-ml-internship-mahid-assignment2/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Method Choice

My lane is Content Refresh Opportunity Scoring.

The goal is to predict whether a page should be prioritised for refresh.

This is a binary classification problem:

1 = Refresh recommended
0 = No refresh recommendation

I selected Logistic Regression as the first model because it is simple, interpretable, and provides probability scores.

I also selected Random Forest because content performance depends on multiple interacting signals and tree-based models can capture non-linear relationships.

The models will be compared against my Week-4 rule-based baseline.

In [1]:
import pandas as pd
import numpy as np


DATA_URL = (
    "https://raw.githubusercontent.com/"
    "flyrank-bih/flyrank-ml-internship-starter/"
    "main/data/raw/content_refresh_anonymized.csv"
)


df = pd.read_csv(DATA_URL)


print(df.shape)

df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
df["target"] = (
    df["trend_direction"]
    .eq("down")
    .astype(int)
)


df["target"].value_counts()

,count
target,
1,16262
0,13738


## Split Design

I use a train-test split to evaluate generalisation.

The model learns patterns from the training data and is evaluated on unseen testing data.

The target variable is separated from input features to avoid leakage.


In [3]:
features = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "content_age_days"
]


X = df[features]

y = df["target"]

In [4]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)

(24000, 8)
(6000, 8)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


log_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000
    ))
])


log_model.fit(
    X_train,
    y_train
)


log_pred = log_model.predict(X_test)


In [6]:
from sklearn.ensemble import RandomForestClassifier


rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


rf_model.fit(
    X_train,
    y_train
)


rf_pred = rf_model.predict(X_test)


In [7]:
from sklearn.metrics import precision_score, recall_score, accuracy_score


results = pd.DataFrame({

"Model":[
    "Week4 Baseline",
    "Logistic Regression",
    "Random Forest"
],

"Precision":[
    0,
    precision_score(y_test, log_pred),
    precision_score(y_test, rf_pred)
],

"Recall":[
    0,
    recall_score(y_test, log_pred),
    recall_score(y_test, rf_pred)
]

})


results

,Model,Precision,Recall
0,Week4 Baseline,0.000000,0.000000
1,Logistic Regression,0.596947,0.769680
2,Random Forest,0.688976,0.753383


In [8]:
baseline_prediction = (
    df.loc[X_test.index,
    "days_since_last_update"]
    >
    df["days_since_last_update"].median()
).astype(int)


baseline_precision = precision_score(
    y_test,
    baseline_prediction
)


results.loc[
0,
"Precision"
]=baseline_precision


results

,Model,Precision,Recall
0,Week4 Baseline,0.541899,0.000000
1,Logistic Regression,0.596947,0.769680
2,Random Forest,0.688976,0.753383


## Error Analysis

The model can make two types of mistakes:

False Positive:
The model recommends refresh but the page does not actually need it.

False Negative:
The model misses a page that should be refreshed.

False positives waste editorial resources.

False negatives may lose traffic opportunities.

The model should therefore be evaluated not only by accuracy but also by business usefulness.

In [9]:
importance = pd.DataFrame({

"feature":features,

"importance":
rf_model.feature_importances_

})


importance.sort_values(
    "importance",
    ascending=False
)

,feature,importance
1,impressions_90d,0.232038
4,avg_position,0.230117
7,content_age_days,0.180047
5,sessions_90d,0.115842
3,ctr,0.084116
2,clicks_90d,0.062849
0,days_since_last_update,0.047883
6,engagement_rate,0.047108


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.